# Astro II Lab 3 — Distance Measures & Thermal History (BBN)

**Goal:** Understand how the concept of "distance" changes in an expanding universe, and explore the early thermal history, specifically Big Bang Nucleosynthesis (BBN).

---

## Part 1: Cosmological Distance Measures

In a static Euclidean space, distance is unambiguous. In our expanding Universe, "distance" depends on how you measure it:

1. **Comoving Distance ($D_C$)**: Factors out the expansion of the universe. Two objects locked in the Hubble flow maintain a constant comoving distance.
2. **Luminosity Distance ($D_L$)**: Relates the observed flux $F$ to the intrinsic luminosity $L$ of a source via $F = L / (4 \pi D_L^2)$. Since photons lose energy via redshift and arrive less frequently, $D_L = (1+z) D_M$.
3. **Angular Diameter Distance ($D_A$)**: Relates the physical transverse size of an object ($l$) to its observed angular size ($\theta$) via $\theta = l / D_A$. In an expanding universe, objects actually appear *larger* in angular size at very high redshift! $D_A = D_M / (1+z)$.

Where $D_M$ is the transverse comoving distance (which is equal to $D_C$ in a flat universe).

**What to do:**
1. Run the cell below to load the interactive tool.
2. Adjust $\Omega_m$ and $\Omega_\Lambda$ to see how the distances change.
3. Observe how $D_A$ turns over and starts decreasing at high redshift!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import ipywidgets as widgets
from IPython.display import display, clear_output

# Constants
c = 299792.458 # km/s

def Ez(z, Om, Ol):
    Ok = 1.0 - Om - Ol
    # To avoid negative values in square root for extreme unphysical parameters
    val = Om*(1+z)**3 + Ok*(1+z)**2 + Ol
    return np.sqrt(np.maximum(val, 1e-10))

def compute_distances(z_max, Om, Ol, H0):
    z_ary = np.linspace(0.001, z_max, 200)
    Ok = 1.0 - Om - Ol
    DH = c / H0
    
    DC = np.zeros_like(z_ary)
    for i, z in enumerate(z_ary):
        val, _ = quad(lambda x: 1.0/Ez(x, Om, Ol), 0, z)
        DC[i] = DH * val
        
    if Ok > 1e-5:
        sqOk = np.sqrt(Ok)
        DM = DH / sqOk * np.sinh(sqOk * DC / DH)
    elif Ok < -1e-5:
        sqOk = np.sqrt(-Ok)
        DM = DH / sqOk * np.sin(sqOk * DC / DH)
    else:
        DM = DC
        
    DL = (1 + z_ary) * DM
    DA = DM / (1 + z_ary)
    
    return z_ary, DC, DL, DA

def plot_distances(Om, Ol, H0, z_max):
    z_ary, DC, DL, DA = compute_distances(z_max, Om, Ol, H0)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(z_ary, DC, label='Comoving Distance $D_C$', lw=2.5, color='#2196F3')
    ax.plot(z_ary, DL, label='Luminosity Distance $D_L$', lw=2.5, color='#F44336')
    ax.plot(z_ary, DA, label='Angular Diameter Distance $D_A$', lw=2.5, color='#4CAF50')
    
    # Also plot D = c*z/H0 (Hubble's law linear approximation) for comparison
    ax.plot(z_ary, c * z_ary / H0, '--', color='gray', label='Linear Hubble Law ($cz/H_0$)')
    
    ax.set_xlabel('Redshift $z$', fontsize=12)
    ax.set_ylabel('Distance [Mpc]', fontsize=12)
    ax.set_xlim(0, z_max)
    ax.set_ylim(0, max(DC[-1], DL[-1]/2, 10000) * 1.1)
    ax.grid(True, alpha=0.3, ls=':')
    ax.legend(fontsize=11)
    ax.set_title('Cosmological Distance Measures', fontsize=14, weight='bold')
    
    plt.tight_layout()
    plt.show()

w_Om = widgets.FloatSlider(value=0.31, min=0.0, max=3.0, step=0.01, description='\u03a9m')
w_Ol = widgets.FloatSlider(value=0.69, min=-2.0, max=3.0, step=0.01, description='\u03a9\u039b')
w_H0 = widgets.FloatSlider(value=70.0, min=30.0, max=100.0, step=1.0, description='H\u2080')
w_zmax = widgets.FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5, description='z_max')

ui = widgets.VBox([
    widgets.HTML('<b>Cosmological Parameters</b>'),
    widgets.HBox([w_Om, w_Ol]),
    widgets.HBox([w_H0, w_zmax])
])

out = widgets.interactive_output(plot_distances, {'Om': w_Om, 'Ol': w_Ol, 'H0': w_H0, 'z_max': w_zmax})
display(ui, out)


Output()

### Discussion points (Distances)
1. **Turnover of $D_A$:** At what redshift does $D_A$ reach a maximum for our Universe ($\Omega_m = 0.31, \Omega_\Lambda = 0.69$)? What does this mean for the apparent size of galaxies at very high redshift?
2. **Linear limit:** At what redshift does the linear Hubble law ($cz / H_0$) start to deviate significantly from the true distance measures?
3. **Empty Universe:** What happens to the distance measures as $\Omega_m \to 0$ and $\Omega_\Lambda \to 0$?


---

## Part 2: Thermal History & Big Bang Nucleosynthesis (BBN)

In the early Universe, the temperature was extremely high, scaling as $T \propto (1+z) \propto 1/a$. The early universe was a state of thermal equilibrium. 
As it expanded and cooled, different epochs of "freeze-out" occurred where interaction rates ($\Gamma$) dropped below the expansion rate ($H$).

One of the most crucial observational probes of the thermal history is **Big Bang Nucleosynthesis (BBN)**, occurring roughly between $T \sim 1\text{ MeV}$ and $T \sim 0.1\text{ MeV}$ (at $t \sim 3$ minutes). 
Protons and neutrons fused to form the light elements: Deuterium (D or $^2$H), Helium-3 ($^3$He), Helium-4 ($^4$He), and a tiny bit of Lithium-7 ($^7$Li).

The final abundances of these elements depend starkly on a single parameter: the **baryon-to-photon ratio**, $\eta$, which is often plotted in units of $10^{-10}$ ($\eta_{10}$). Measurements of the Cosmic Microwave Background (CMB) pin down this value extraordinarily well.

### The Schramm Plot
The figure below is a simplified version of the famous **Schramm Plot**, illustrating the predicted primordial abundances of light elements as a function of the baryon-to-photon ratio $\eta$ (or equivalently, the baryon density $\Omega_b h^2$).

**What to do:**
1. Adjust the $\eta_{10}$ slider. 
2. See how the abundances of Helium-4 (mass fraction $Y_p$) and the trace elements (number fractions relative to hydrogen) change.
3. The vertical band indicates the region consistent with CMB observations and actual observed abundances, representing a huge triumph for the Hot Big Bang model!


In [ ]:
# Interactive Schramm plot (Approximate theoretical abundances)

def plot_bbn(eta_10_choice):
    # eta_10 values for the x-axis
    eta_10 = np.logspace(-1, 1.3, 200)
    
    # APPROXIMATE fitting formulas for illustration of BBN trends
    # (These capture the qualitative behavior: steep decline of D/H, weak log increase of Yp, Li7 'valley')
    # Mass fraction of ^4He
    Y_p = 0.245 + 0.014 * np.log10(eta_10 / 6.0)
    Y_p = np.clip(Y_p, 0.1, 0.35) 
    
    # Number fraction of D/H
    D_H = 2.5e-5 * (eta_10 / 6.0)**-1.6
    
    # Number fraction of ^3He/H
    He3_H = 1.0e-5 * (eta_10 / 6.0)**-0.5
    
    # Number fraction of ^7Li/H (shows the characteristic 'valley' due to distinct production channels)
    Li7_H = 4e-10 * (eta_10 / 6.0)**2.2 + 1.2e-10 * (eta_10 / 6.0)**-2.5
    
    fig, ax1 = plt.subplots(figsize=(11, 7))
    
    # Plotting on Log-Log or Log-Linear axes
    ax1.set_xscale('log')
    
    # Y_p uses a linear scale, usually plotted at the top or on a separate panel. 
    # For simplicity, we'll plot Yp on a twin axis so the ranges are distinct.
    ax2 = ax1.twinx()
    
    # Yp (He-4)
    line1, = ax2.plot(eta_10, Y_p, label='$^4$He ($Y_p$, mass fraction)', color='purple', lw=2.5)
    ax2.set_ylabel('Mass Fraction (for $^4$He)', fontsize=12, color='purple')
    ax2.set_ylim(0.20, 0.28)
    ax2.tick_params(axis='y', labelcolor='purple')
    
    # Trace elements (log scale below)
    line2, = ax1.plot(eta_10, D_H, label='D/H', color='blue', lw=2.5)
    line3, = ax1.plot(eta_10, He3_H, label='$^3$He/H', color='green', lw=2.5)
    line4, = ax1.plot(eta_10, Li7_H, label='$^7$Li/H', color='orange', lw=2.5)
    
    ax1.set_yscale('log')
    ax1.set_ylabel('Number Ratio relative to H', fontsize=12)
    ax1.set_ylim(1e-10, 1e-3)
    ax1.set_xlim(0.1, 20)
    
    # Highlight the current choice
    ax1.axvline(eta_10_choice, color='red', linestyle='--', lw=2, alpha=0.8)
    
    # Calculate values at choice
    Y_p_val = 0.245 + 0.014 * np.log10(eta_10_choice / 6.0)
    D_H_val = 2.5e-5 * (eta_10_choice / 6.0)**-1.6
    He3_H_val = 1.0e-5 * (eta_10_choice / 6.0)**-0.5
    Li7_H_val = 4e-10 * (eta_10_choice / 6.0)**2.2 + 1.2e-10 * (eta_10_choice / 6.0)**-2.5
    
    # Display the values
    info_text = (f"\u03b7\u2081\u2080 = {eta_10_choice:.1f}\n"
                 f"$Y_p$ = {Y_p_val:.4f}\n"
                 f"D/H = {D_H_val:.2e}\n"
                 f"$^3$He/H = {He3_H_val:.2e}\n"
                 f"$^7$Li/H = {Li7_H_val:.2e}")
                 
    ax1.text(0.05, 0.05, info_text, transform=ax1.transAxes, fontsize=11,
            bbox=dict(facecolor='white', alpha=0.9, edgecolor='black'))
    
    # Combine legends from both axes and place OUTSIDE the plot to prevent overlapping with content/titles
    lines = [line1, line2, line3, line4]
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left', bbox_to_anchor=(1.1, 1), fontsize=11)
    
    # Add CMB constraint band
    cmb_eta = 6.1  # Approx Planck value for eta_10
    ax1.axvspan(cmb_eta*0.9, cmb_eta*1.1, color='gray', alpha=0.2, label='CMB Constraint')
    ax1.text(cmb_eta, 5e-5, 'CMB\nConstraint', horizontalalignment='center', color='gray', fontweight='bold')
    
    # Secondary x-axis for Omega_b h^2
    def eta_to_ombh2(eta):
        # Omega_b h^2 ≈ eta_10 / 274
        return eta / 273.9
    def ombh2_to_eta(ombh2):
        return ombh2 * 273.9
        
    secax = ax1.secondary_xaxis('top', functions=(eta_to_ombh2, ombh2_to_eta))
    secax.set_xlabel('$\Omega_b h^2$', fontsize=12)
    
    ax1.set_xlabel('Baryon-to-photon ratio, $\eta \\times 10^{10}$', fontsize=12)
    plt.title('Primordial Abundances (BBN)', fontsize=14, weight='bold', pad=10)
    
    ax1.grid(True, which="both", ls="-", alpha=0.2)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.75, top=0.88) # Provide more room for legend and twin axes titles
    plt.show()

w_eta = widgets.FloatLogSlider(value=6.1, base=10, min=-1, max=1.3, step=0.01, description='\u03b7\u2081\u2080')

ui_bbn = widgets.VBox([
    widgets.HTML('<b>Adjust Baryon-to-Photon Ratio</b> (Current CMB value is ~6.1)'),
    w_eta
])

out_bbn = widgets.interactive_output(plot_bbn, {'eta_10_choice': w_eta})
display(ui_bbn, out_bbn)


Output()

### Discussion points (Thermal History / BBN)
1. **The Deuterium "Bottleneck":** Deuterium is an intermediate step in forming Helium. Why is Deuterium considered an excellent "baryometer"? (Hint: Look at how steep the D/H curve is as a function of $\eta$!)
2. **Lithium Problem:** If you look at the observed constraints for $^7$Li today, they often fall lower than the theoretical prediction (the dip in the Schramm plot). This is the famous "Cosmological Lithium Problem." Discuss what might cause this discrepancy inside stars.
3. **Neutron Half-Life:** If free neutrons decayed significantly faster or slower, how would that affect $Y_p$ (the $^4$He abundance)?


---
*This notebook utilizes approximate fitting equations to standard BBN networks for pedagogical visualization purposes. Advanced analyses rely on full Boltzmann and nucleosynthesis codes like AlterBBN or PArthENoPE.*
